In [ ]:
'''
기획 : 스마트 워치 만들기

1. 제스처 무접촉 메뉴 보드 출력하기(제스처 사용 -> OLED 출력 메뉴 이동)
2. 버튼으로 메뉴 입출력 (OLED에서 나오는 메뉴에서 0은 enter, 1은 back을 사용)
3. 메뉴 목록
    1) 주변 환경 체크 - 온습도 / 밝기 / 소음을 체크해서 좋음/주의/경고 로 OLED출력 및 led로 점수 출력
    2) 배터리 잔량 확인 - 배터리 잔량 확인하기 - led로 잔량 표기(최대 70%~최소 10%로 20%아래는 buzzor 경고)
    3) 스탑워치 - 책 뒤에 있는 내용 사용해서 스탑워치 구상하기 - OLED, 버튼으로 구성, 0번 버튼으로 시작 멈추기하고 1번으로 뒤로 가기
    4) 타이머 - 분과 초 선택은 무접촉 제스처로 좌우 이동하고, poten값으로 최대 0분부터 59분 까지, 0초부터 59초 까지 입력가능, 버튼 0으로 입력, 1로 나가기
            -> 과정 : 제스처 무접촉 메뉴 보드에서 0번입력으로 들어옴 -> 분 입력으로 입장 
            -> poten의 실시간 값으로 0번 입력 시 분 최종 결정, 이후 자동으로 초 단위 입력으로 넘아감.
            -> poten의 실시간 값으로 0번 입력 시 초 최종 결정.
            -> 이후 시작할거면 0, 뒤로 가면 1 
            -> 다시 들어가면 분, 초 초기화
    5) 야간 경고등 : 해당 스마트 워치를 자전거나 가방에 묶는다고 생각하고, 
                    뒤에서 거리가 가까울 수록 led가 점점 누적으로 많이 켜지다가 7개가 된 후 일정값이 더 가까이 가면, 
                    buzz에서 알람이 울리고, led가 좌우 왔다갔다하며 알림이 나옴.
                    끄는 기능은 0번 버튼, 모든 기능 종료 후 뒤로가기는 1번의 버튼 1번으로 정리
    6) 모든 기능 종료


'''

import pop
from pop import delay
# 초기 클래스 지정
led = pop.Leds()                # 불빛 클래스               # led[1~7].on(),off() / led.allOn(), allOff()
btn = pop.Switches()            # 스위치 클래스             # btn[0].read = false -> 누름 / true -> 안 누름
psd = pop.Psd()                 # 적외선 클래스             # psd.readAverage() : 센서값을 ADC값으로 변환 . psd.calcDist(val) : 전달받은 인자를 거리값으로 반환
cds = pop.Cds()                 # 조도 센서 클래스          # cds.readAverage() : 광량 측정하여 lux값으로 변환
sound = pop.Sound()             # 소음측정 클래스           # sound.read() :소리크기 / sound.readAverage() :센서값 측정 후 ADC로 변환
poten = pop.Potentiometer()    # 측정 클래스               # poten.read() : ADC 결과 값 반환 / poten.readVoltAverage() : 전압값을 측정 후 출력
buzz = pop.PiezoBuzzer()        # 버저 클래스               # buzz.setTemp(n): 빠르기 값 / buzz.tone(a, b, c) : 옥타브(a)의 음계(b)를 지속시간(c)만큼 출력
pixel = pop.PixelDisplay()      # 픽셀 디스플레이 클래스     # X 
sht = pop.Sht20()               # 온습도 클래스             # sht.readTemp() : 온도 측정 / sht.readHumi() : 습도측정
gs = pop.Gesture()              # 움직임 센서 클래스         # gs.isAvailable() : 사용 여부 / read() : 감지된 동작 방향 번호로 반환 , readStr() : 감지된 동장 방향 string으로 반환
display = pop.Oled()            # 화면 클래스               # oled.init() : 초기화 / oled.print() , oled.drawCircle,fillCircle(x좌표, y좌표, 반지름, 색깔) , 
                                                           # drawLine(x0, y0, x1, y1, color) , fillCircle(x0, y0, r, color) , drawRect(x, y, w, h, color), fillRect(x, y, w, h, color), drawRoundRect(x, y, w, h, r, color) , fillRoundRect(x, y, w, h, r, color)
                                                            # drawTriangle(x0, y0, x1, y1, x2, y2, color) , fillTriangle(x0, y0, x1, y1, x2, y2, color), drawBitmap(x, y, bitmap, w, h, color)
                                                            # setCursor(x, y) : 글자 출력 시 좌표 설정 , setTextSize(s) : 글자 출력시 사용할 글자 크기 설정 , setTextColor(c) : 글자 출력시 글자 색상 설정
                                                            # setTextColorWithBg(c, b) : 글자 출력시 사용할 글자 색상(c, Oled.black, white)과 배경색상(b, Oled.black, white)
                                                            # display.display() : 세팅된 display를 최종 출력 / display.clearDisplay() : OLed 화면 삭제
from time import time

# 전역 변수 지정 및 초기화
app_running = True # 코드 종료 변수
led.allOff()
display.init()
display.clearDisplay()
menu_y = [8, 16, 24, 32, 40, 48]

#-------------------함수 모음집--------------
def run_action(action_table, level):
    fn = action_table.get(level)
    if fn is not None:
        fn()
    return 

def sw0_env():
    def temp_score(t):
        if 18 <= t <= 22: return 100
        if 17 <= t <= 23: return 85
        if 16 <= t <= 24: return 70
        if 15 <= t <= 26: return 40
        return 0

    def humi_score(h):
        if 40 <= h <= 60: return 100
        if 35 <= h <= 65: return 85
        if 30 <= h <= 70: return 70
        if 25 <= h <= 75: return 40
        return 0

    def raw_to_db(raw):
        if raw <= 500: return 30
        if raw <= 800: return 35
        if raw <= 1100: return 40
        if raw <= 1400: return 45
        if raw <= 1700: return 50
        if raw <= 2000: return 55
        if raw <= 2400: return 60
        if raw <= 2900: return 65
        if raw <= 3500: return 70
        return 75

    def sound_score(db):
        if 40 <= db <= 50: return 100
        if 35 <= db <= 55: return 85
        if 30 <= db <= 60: return 70
        if 25 <= db <= 65: return 40
        return 0

    def set_led(n):
        for i in range(7):
            if i < n: led[i].on()
            else: led[i].off()

    # 좌표 고정
    y_temp, y_humi, y_db, y_score = 12, 22, 32, 42
    x_val = 46

    # 고정 UI (한 번만 그림)
    display.clearDisplay()
    display.setTextSize(1)
    display.setCursor(0, 0);      display.print("Env Check")
    display.setCursor(0, y_temp);  display.print("Temp :")
    display.setCursor(0, y_humi);  display.print("Humi :")
    display.setCursor(0, y_db);    display.print("dB   :")
    display.setCursor(0, y_score); display.print("Score:")

    # 단위/고정 표기(한 번만 그림)
    display.setCursor(92, y_temp);  display.print("C")
    display.setCursor(92, y_humi);  display.print("%")
    display.setCursor(84, y_db);    display.print("dB(A)")
    display.setCursor(82, y_score); display.print("/100")
    display.setCursor(0, 54);       display.print("Env Score : led(0~7)")
    
    
    display.display()

    current_led = 0
    while True:
        # 뒤로가기: SW1
        if not btn[1].read():
            while not btn[1].read():
                delay(20)
            led.allOff()
            display.clearDisplay()
            return
        
        
        t = float(sht.readTemp())
        h = float(sht.readHumi())
        db = raw_to_db(int(sound.readAverage()))

        score = int(round(
            temp_score(t) * 0.35 +
            humi_score(h) * 0.35 +
            sound_score(db) * 0.30
        ))
        target_led = max(0, min(7, round(score * 7 / 100)))

        if current_led < target_led: current_led += 1
        elif current_led > target_led: current_led -= 1
        set_led(current_led)

        # 숫자 영역만 지움 (단위/마지막 줄은 유지)
        display.fillRect(x_val, y_temp, 34, 8, display.BLACK)
        display.fillRect(x_val, y_humi, 34, 8, display.BLACK)
        display.fillRect(x_val, y_db,   34, 8, display.BLACK)
        display.fillRect(x_val, y_score,34, 8, display.BLACK)

        # 우측 정렬 출력
        display.setCursor(x_val, y_temp);  display.print("{:>5.1f}".format(t))
        display.setCursor(x_val, y_humi);  display.print("{:>5.1f}".format(h))
        display.setCursor(x_val, y_db);    display.print("{:>3d}".format(db))
        display.setCursor(x_val, y_score); display.print("{:>3d}".format(score))
        display.display()

        delay(120)


def sw0_battery(): 
    print("SW0 -> Battery")
#--------------------------------------------------------
    # 맨처음 켤때 배터리가 50%라고 인식
    battery_now = 50.0
    cds_datumPoint = 1900.0
    psd_datumPoint = 16.0
    list_status = ["normal", " low", "charge"]# low는 글자가 짧아서 일부러 넣은것.
    battery_status = list_status[0]
    print(f"battery_status : {battery_status}")
#--------------------------------------------------------
    # 조도 센서와 적외선 센서 사용
    # 특수 상황에 사운드 울리게 만들기
    buzz.setTempo(120)
    
    # 최초 화면 구성
    display.clearDisplay()
    display.setTextSize(1)
    display.setCursor(0, 0)
    display.print("Battery Situation")
    #display.setCursor(0, 10)
    display.drawRect(0,10,50,50,display.WHITE)
    display.setCursor(55, 25)
    display.print("Level:")
    display.setCursor(55, 45)
    display.print("state:")
    
    # 위치 확인용
    # display.setCursor(92, 25)
    # display.print(f"{battery_now}%")
    # display.setCursor(92, 45)
    # display.print(battery_status)
    
    # 화면 출력
    display.display()
    
        
    #     #delay(1000)
    #     #print(charge)
    #     #print(batteryConsumption)
    #last_tick = time()  # 1초 주기 타이머

     # [추가] 상황 통제 변수 초기화
    prev_level = -1          # 이전 표시 배터리(%)
    prev_status = None       # 이전 상태 문자열
    first_draw = True        # 최초 1회 강제 출력

    while True:
        # 뒤로가기 버튼은 즉시 반응
        if not btn[1].read():
            while not btn[1].read():
                delay(20)
            led.allOff()
            display.clearDisplay()
            return
        


        #-------------------------------------------------------
        # PSD를 통한 배터리 사용
        val = psd.readAverage()
        batteryConsumption = psd.calcDist(val)
        if batteryConsumption <= psd_datumPoint:
            battery_now -= 5.0

        # 충전 의미
        charge = cds.readAverage()
        # if charge >= cds_datumPoint:
        #     battery_now += 10.0

        

        # 상태 결정 (Charge 우선)
        if charge >= cds_datumPoint:
            battery_status = list_status[2]   # charge
            battery_now += 10.0
            
        elif battery_now <= 15.0:
            battery_status = list_status[1]   # low
            buzz.tone(4,8,4)
        else:
            battery_status = list_status[0]   # normal
        
        # 배터리 범위 고정
        # 배터리의 모든 값 변동을 받은 이후 지정을 해줘야 오류가 없음.
        if battery_now < 0.0:
            battery_now = 0.0
        elif battery_now > 100.0:
            battery_now = 100.0
            
            
        
        # [추가] 상황 통제 변수 조건
        level_now = int(round(battery_now))        
        changed = first_draw or (level_now != prev_level) or (battery_status != prev_status)

        if changed:
            # 동적 영역 지우기(계산 값이 변하는 공간)
            #---------------------------------------------------------
            # 해당 사각형 영역만 지움: 아이콘 + 값 영역
            display.fillRect(2, 12, 46, 46, display.BLACK)     # 아이콘 영역
            display.fillRect(90, 25, 36, 8, display.BLACK)     # Level 값 영역
            display.fillRect(90, 45, 36, 8, display.BLACK)     # state 값 영역
            # 지우는 코드가 없지만, 검은색으로 칠하는것을 지우개로 대체
            # 상태별 이미지
            if battery_status == list_status[0]:   # normal
                display.fillRect(10, 18, 30, 30, display.WHITE)

            elif battery_status == list_status[1]: # low
                display.fillTriangle(10, 18, 40, 18, 25, 48, display.WHITE)

            elif battery_status == list_status[2]: # charge
                display.fillTriangle(25, 15, 15, 40, 25, 40, display.WHITE)
                display.fillTriangle(25, 28, 35, 28, 25, 52, display.WHITE)

            # 값 출력
            display.setCursor(90, 25)
            display.print("{:3d}%".format(int(round(battery_now))))
            display.setCursor(90, 45)
            display.print("{:<6}".format(battery_status))

            display.display()
             # [추가] 상황 통제 변수 갱신
            prev_level = level_now
            prev_status = battery_status
            first_draw = False

        # # 1초마다만 갱신
        # if time() - last_tick < 1.0:
        #     delay(20)
        #     continue
        # last_tick = time()
        delay(1000)
    
    
def sw0_stopwatch():
    print("SW0 -> StopWatch")

    # 최초 화면
    display.clearDisplay()
    display.setTextSize(1)
    display.setCursor(35, 0)
    display.print("StopWatch")

    display.drawRect(10, 10, 100, 35, display.WHITE)
    display.setCursor(0, 50)
    display.print("SW0:Start SW1:Back")

    display.setTextSize(2)
    display.setCursor(30, 22)
    display.print("00:00")
    display.display()

    # 시간 자리 변수
    timer_sec_one = 0
    timer_sec_ten = 0
    timer_min_one = 0
    timer_min_ten = 0

    is_running = False
    prev_btn0 = True
    prev_btn1 = True
    last_tick = time()

    display.setTextSize(2)
    display.setCursor(54, 22)
    # 중간 지점은 고정
    display.print(":")      
    display.display()

    prev_min_ten = -1
    prev_min_one = -1
    prev_sec_ten = -1
    prev_sec_one = -1

    while True:
        btn_0 = btn[0].read()   # 눌림=False
        btn_1 = btn[1].read()

        # SW1: 뒤로가기
        if prev_btn1 and (not btn_1):
            while not btn[1].read():
                delay(20)
            led.allOff()
            display.clearDisplay()
            display.display()
            return

        # SW0: 시작/정지 
        if prev_btn0 and (not btn_0):
            while not btn[0].read():
                delay(20)

            is_running = not is_running
            if is_running:
                last_tick = time() 

            display.fillRect(0, 50, 128, 12, display.BLACK)
            display.setTextSize(1)
            display.setCursor(0, 50)
            if is_running:
                display.print("SW0:Stop  SW1:Back")
            else:
                display.print("SW0:Start SW1:Back")
            display.display()

        if is_running and (time() - last_tick >= 1.0):
            last_tick = time()  

            timer_sec_one += 1
            if timer_sec_one == 10:
                timer_sec_one = 0
                timer_sec_ten += 1
            if timer_sec_ten == 6:
                timer_sec_ten = 0
                timer_min_one += 1
            if timer_min_one == 10:
                timer_min_one = 0
                timer_min_ten += 1
            if timer_min_ten == 10:
                timer_min_ten = 9
                timer_min_one = 9
                timer_sec_ten = 5
                timer_sec_one = 9
                is_running = False

            need_display = False
            display.setTextSize(2)

            if timer_min_ten != prev_min_ten:
                display.fillRect(30, 22, 12, 16, display.BLACK)
                display.setCursor(30, 22)
                display.print(timer_min_ten)
                prev_min_ten = timer_min_ten
                need_display = True

            if timer_min_one != prev_min_one:
                display.fillRect(42, 22, 12, 16, display.BLACK)
                display.setCursor(42, 22)
                display.print(timer_min_one)
                prev_min_one = timer_min_one
                need_display = True

            if timer_sec_ten != prev_sec_ten:
                display.fillRect(66, 22, 12, 16, display.BLACK)
                display.setCursor(66, 22)
                display.print(timer_sec_ten)
                prev_sec_ten = timer_sec_ten
                need_display = True

            if timer_sec_one != prev_sec_one:
                display.fillRect(78, 22, 12, 16, display.BLACK)
                display.setCursor(78, 22)
                display.print(timer_sec_one)
                prev_sec_one = timer_sec_one
                need_display = True

            if need_display:
                display.display()

        prev_btn0 = btn_0
        prev_btn1 = btn_1
        delay(20)
        

def sw0_timer():
    print("SW0 -> Timer")
    # 메인에서 넘어온 SW0 눌림 잔여 입력 제거
    while not btn[0].read():
        delay(20)
    while not btn[1].read():
        delay(20)
    
    prev_btn0 = btn[0].read()
    prev_btn1 = btn[1].read()
    # 타이머 목록
    timer_text_list = ["05:00", "01:00", "00:10"]
    timer_sec_list = [300, 60, 10]

    if len(timer_text_list) != len(timer_sec_list):
        print("Timer list length mismatch")
        return

    timer_count = len(timer_text_list)
    buzz.setTempo(120)

    selected_index = 0
    prev_btn0 = True
    prev_btn1 = True

    is_select_mode = True
    is_run_mode = False
    is_alarm_mode = False

    remain_sec = 0
    last_tick = time()
    alarm_tick = time()

    prev_draw_index = -1
    prev_draw_remain = -1

    while True:
        btn_0 = btn[0].read()   # 눌림=False
        btn_1 = btn[1].read()

        # ---------------------------
        # 1) 선택 모드
        # ---------------------------
        if is_select_mode:
            # SW1: 메인 복귀
            if prev_btn1 and (not btn_1):
                while not btn[1].read():
                    delay(20)
                display.clearDisplay()
                display.display()
                return

            # poten -> index
            raw_value = poten.read()
            now_index = (raw_value * timer_count) // 4096
            if now_index < 0:
                now_index = 0
            elif now_index > timer_count - 1:
                now_index = timer_count - 1

            # 0.3초 안정화
            if now_index != selected_index:
                delay(300)
                raw_value = poten.read()
                now_index = (raw_value * timer_count) // 4096
                if now_index < 0:
                    now_index = 0
                elif now_index > timer_count - 1:
                    now_index = timer_count - 1
                selected_index = now_index

            # 선택 화면 (index 바뀔 때만)
            if selected_index != prev_draw_index:
                display.clearDisplay()
                display.setTextSize(1)
                display.setTextColor(display.WHITE)
                display.setCursor(40, 0)
                display.print("Timer")

                for i in range(timer_count):
                    x = 2 + i * 42
                    y = 22
                    w = 40
                    h = 24

                    display.drawRect(x, y, w, h, display.WHITE)

                    if i == selected_index:
                        display.fillRect(x + 1, y + 1, w - 2, h - 2, display.WHITE)
                        display.setTextColor(display.BLACK)
                        display.setCursor(x + 5, y + 8)
                        display.print(timer_text_list[i])
                        display.setTextColor(display.WHITE)
                    else:
                        display.setCursor(x + 5, y + 8)
                        display.print(timer_text_list[i])

                display.display()
                prev_draw_index = selected_index

            # SW0: 타이머 시작
            if prev_btn0 and (not btn_0):
                while not btn[0].read():
                    delay(20)

                remain_sec = timer_sec_list[selected_index]
                prev_draw_remain = -1
                last_tick = time()

                # 실행 화면 1회 출력
                display.clearDisplay()
                display.setTextSize(1)
                display.setTextColor(display.WHITE)
                display.setCursor(40, 0)
                display.print("Timer")
                display.drawRect(10, 20, 108, 30, display.WHITE)
                display.setCursor(8, 54)
                display.print("SW1:Back")
                display.display()

                is_select_mode = False
                is_run_mode = True
                is_alarm_mode = False

        # ---------------------------
        # 2) 실행 모드
        # ---------------------------
        elif is_run_mode:
            # SW1: 메인 복귀
            if not btn[1].read():
                while not btn[1].read():
                    delay(20)
                display.clearDisplay()
                display.display()
                return

            # 시간 표시(변할 때만)
            if remain_sec != prev_draw_remain:
                mm = remain_sec // 60
                ss = remain_sec % 60

                display.fillRect(18, 24, 92, 22, display.BLACK)
                display.setTextSize(2)
                display.setTextColor(display.WHITE)
                display.setCursor(28, 28)
                display.print("{:02d}:{:02d}".format(mm, ss))
                display.display()
                prev_draw_remain = remain_sec

            # 1초 카운트다운
            if time() - last_tick >= 1.0:
                last_tick += 1.0
                if remain_sec > 0:
                    remain_sec -= 1

            # 종료 -> 알람 모드
            if remain_sec <= 0:
                display.clearDisplay()
                display.setTextSize(1)
                display.setTextColor(display.WHITE)
                display.setCursor(40, 0)
                display.print("Timer")
                display.drawRect(10, 20, 108, 30, display.WHITE)
                display.setTextSize(2)
                display.setCursor(28, 28)
                display.print("00:00")
                display.setTextSize(1)
                display.setCursor(4, 54)
                display.print("SW0:Retry  SW1:Back")
                display.display()

                alarm_tick = time()
                is_select_mode = False
                is_run_mode = False
                is_alarm_mode = True

        # ---------------------------
        # 3) 종료 알람 모드
        # ---------------------------
        elif is_alarm_mode:
            # SW1: 메인 복귀
            if not btn[1].read():
                while not btn[1].read():
                    delay(20)
                display.clearDisplay()
                display.display()
                return

            # SW0: 선택 화면으로 복귀
            if not btn[0].read():
                while not btn[0].read():
                    delay(20)
                is_select_mode = True
                is_run_mode = False
                is_alarm_mode = False
                prev_draw_index = -1
                prev_btn0 = btn[0].read()
                prev_btn1 = btn[1].read()
                delay(20)
                continue

            # 1초마다 버저
            if time() - alarm_tick >= 1.0:
                alarm_tick += 1.0
                buzz.tone(4, 8, 4)

        prev_btn0 = btn_0
        prev_btn1 = btn_1
        delay(20)

        
def sw0_night(): 
    def led_set(led_count=0, _delay=100, onoff = True, reversed=False):
        # delay = led 딜레이
        # onoff = True 는 불 키기 / False는 불 끄기
        # reversed = 켜지고 꺼지는 순서가 정순(True), 역순(False) 체크
        # 최대, 최소 오류 감지
        led_max = 7
        if led_count<0:
            led_count=0
        elif led_count>led_max:
            led_count=led_max
        
        # 시작 전 버그 방지를 위해 led 전부 꺼놓기
        led.allOff()
        
        if onoff:#led 키기
            if not reversed: # 정순으로 키기
                for i in range(0, led_count, 1):
                    led[i].on()
                    delay(_delay)
            else: # 역순으로 키기
                for i in range(led_count, 0, -1):
                    led[i].on()
                    delay(_delay)        
        else: #led 끄기
            if not reversed: # 정순으로 끄기
                for i in range(0,led_count, 1):
                    led[i].off()
                    delay(_delay)
            else:
                for i in range(led_count, 0, -1):
                    led[i].off()
                    delay(_delay)
         
    print("SW0 -> Night")
    while not btn[0].read():
        delay(20)
    while not btn[1].read():
        delay(20)
    # 최초 화면
    display.clearDisplay()
    display.setTextSize(1)
    display.setCursor(35, 0)
    display.print("Night Light")
    display.setCursor(0, 20)
    display.print("mode:")
    mode = 0
    prev_mode = 0
    mode_list = ["led to cds", "Led All On", "Led Siren", "Led Wave", "Led to Poten", "led to Psd"]
    buzz.setTempo(120)
    prev_btn0 = True  # True: 안 눌림, False: 눌림
    prev_level = -1   # 최초 1회 강제 갱신용
    while True:
        # 뒤로가기: SW1
        if not btn[1].read():
            while not btn[1].read():
                delay(20)
            led.allOff()
            display.clearDisplay()
            return
        
        
        illuminance = cds.readAverage()
        btn_0 = btn[0].read()
        
        # 버튼 0 한 번 눌렀을 때만 모드 변경
        if prev_btn0 and (not btn_0):
            mode = (mode + 1) % len(mode_list)
            print("mode ->", mode_list[mode])
            prev_level = -1   # 최초 1회 강제 갱신용
        prev_btn0 = btn_0
        current_mode = mode_list[mode]
        
        
        # ---------------------------------------------------------
        # 모드 1---------------------------------------------------
        if current_mode == "led to cds":
            # 뒤로가기: SW1
            if not btn[1].read():
                while not btn[1].read():
                    delay(20)
                led.allOff()
                display.clearDisplay()
                return
            if prev_mode != current_mode:
                # display.clearDisplay()
                display.fillRect(30, 20, 80, 15, display.BLACK)
                display.setTextSize(1)
                display.setCursor(30, 20)
                display.print("led to cds")
                #display.fillRect(0, 50, 100, 15, display.WHITE)
            # 조도 -> LED 개수
            if illuminance < 300:
                led_n = 8
            elif illuminance < 600:
                led_n = 7
            elif illuminance < 900:
                led_n = 6
            elif illuminance < 1200:
                led_n = 5
            elif illuminance < 1500:
                led_n = 4
            elif illuminance < 1800:
                led_n = 3
            elif illuminance < 2100:
                led_n = 2
            else:
                led_n = 1    
            led_set(led_n, 10, True, True)
            
            # 광과민성 주의 경고문
            if led_n<=3:
                display.fillRect(0, 40, 160, 20, display.BLACK)
                display.setCursor(0, 40)
                display.print("Waring Photosensitivity syndrome")
            else:
                display.fillRect(0, 40, 160, 20, display.BLACK)
                display.setCursor(0, 40)
                display.print("led to cds")
            prev_mode = current_mode  
        
        elif current_mode == "Led All On":
            # 뒤로가기: SW1
            if not btn[1].read():
                while not btn[1].read():
                    delay(20)
                led.allOff()
                display.clearDisplay()
                return
            
            if prev_mode != current_mode:
                # display.clearDisplay()
                display.fillRect(30, 20, 80, 15, display.BLACK)
                display.setTextSize(1)
                display.setCursor(30, 20)
                display.print("Led All On")
                #display.fillRect(0, 50, 100, 15, display.WHITE)
            
            led_set(7, 0, True, True)
            display.fillRect(0, 40, 160, 20, display.BLACK)
            display.setCursor(0, 40)
            display.print("Led All On")
            prev_mode = current_mode
            pass
        
        elif current_mode == "Led Siren":
            # 뒤로가기: SW1
            if not btn[1].read():
                while not btn[1].read():
                    delay(20)
                led.allOff()
                display.clearDisplay()
                return
            
            if prev_mode != current_mode:
                display.fillRect(30, 20, 80, 15, display.BLACK)
                display.setTextSize(1)
                display.setCursor(30, 20)
                display.print("Led Siren")
            
            
            led.allOff()
            for i in range(0, 7, 2):#2, 4, 6
                led[i].on()
                delay(10)
            buzz.tone(4, 8, 8)
            delay(200)
            led.allOff()
            for i in range(1, 8, 2):
                led[i].on()
                delay(10)
            prev_mode = current_mode
            pass
        
        elif current_mode == "Led Wave":
            # 뒤로가기: SW1
            if not btn[1].read():
                while not btn[1].read():
                    delay(20)
                led.allOff()
                display.clearDisplay()
                return
            if prev_mode != current_mode:
                display.fillRect(30, 20, 80, 15, display.BLACK)
                display.setTextSize(1)
                display.setCursor(30, 20)
                display.print("Led Wave")
            for i in range(0, 8, 1):
                led[i].on()
                delay(100)
            for i in range(7, -1, -1):
                led[i].off()
                delay(100)

            prev_mode = current_mode    
            pass
        
        elif current_mode == "Led to Poten":
            # 뒤로가기: SW1
            if not btn[1].read():
                while not btn[1].read():
                    delay(20)
                led.allOff()
                display.clearDisplay()
                return
            if prev_mode != current_mode:
                display.fillRect(30, 20, 80, 15, display.BLACK)
                display.setTextSize(1)
                display.setCursor(30, 20)
                display.print("Led to Poten")
            

            potenRead = poten.read()              # 0~4095
            level = (potenRead * 8) // 4096       # 0~7
            if level != prev_level:
                led.allOff()
                for i in range(level + 1):
                    led[i].on()
                prev_level = level
            prev_mode = current_mode 
            delay(30)
            
            
            
            pass
        elif current_mode == "led to Psd":
            # 뒤로가기: SW1
            if not btn[1].read():
                while not btn[1].read():
                    delay(20)
                led.allOff()
                display.clearDisplay()
                return
            if prev_mode != current_mode:
                display.fillRect(30, 20, 80, 15, display.BLACK)
                display.setTextSize(1)
                display.setCursor(30, 20)
                display.print("Led to Psd")
            val = psd.read()
            dist = psd.calcDist(val) # 
            
            if dist >= 16:
                dist = 16
            elif dist <= 8:
                dist = 8
            led.allOff()
            for i in range(0, (int(dist)-8)):
                led[i].on()
            
            display.fillRect(0, 40, 160, 20, display.BLACK)
            display.setCursor(0, 40)
            display.print("psd : {:2f}".format(dist))    
            delay(100)
            
            prev_mode = current_mode 
            pass
        
        delay(20)
         

def sw0_exit():     
    # print("SW0 -> Exit")
    # led.allOff()
    # display.clearDisplay()
    # display.display()
    
    # # 코드 완전 종료 선언
    # raise SystemExit
    global app_running
    print("SW0 -> Exit")
    app_running = False
    return
    
#--------------------------------------------------------------------

def sw1_env(): 
    print("SW1 -> ENV Back")
    
def sw1_battery(): 
    print("SW1 -> Battery Back")
    
def sw1_stopwatch(): 
    print("SW1 -> StopWatch Back")
    
def sw1_timer(): 
    print("SW1 -> Timer Back")
    
def sw1_night(): 
    print("SW1 -> Night Back")
    
def sw1_exit(): 
    print("SW1 -> Exit Back")
    
#--------------------------------------------------------------------
SW0_ACTIONS = {
    1: sw0_env,
    2: sw0_battery,
    3: sw0_stopwatch,
    4: sw0_timer,
    5: sw0_night,
    6: sw0_exit,
}
SW1_ACTIONS = {
    1: sw1_env,
    2: sw1_battery,
    3: sw1_stopwatch,
    4: sw1_timer,
    5: sw1_night,
    6: sw1_exit,
}
#----------------------------------------------------------------------------------

class mainController:
    def read_level():
        menu_count = len(SW0_ACTIONS)      # 메뉴 개수 자동 반영
        raw = poten.read()
        level = (raw * menu_count) // 4096 + 1
        if level < 1:
            level = 1
        elif level > menu_count:
            level = menu_count
        return level

    def draw_main_static():
        display.clearDisplay()
        display.setTextSize(1)
        display.setCursor(0, 5)
        display.print("1.ENV check\n2.Battery\n3.StopWatch\n4.Timer\n5.Night\n6.Exit")
        for y in menu_y:
            display.drawCircle(100, y, 4, display.WHITE)  # 비선택 원
        display.display()

    def draw_cursor(old_level, new_level):
        # 이전 선택 원 -> 비선택 원으로 복구
        if old_level is not None:
            y_old = menu_y[old_level - 1]
            display.fillCircle(100, y_old, 3, display.BLACK)
            display.drawCircle(100, y_old, 4, display.WHITE)

        # 새 선택 원 -> 채움
        y_new = menu_y[new_level - 1]
        display.fillCircle(100, y_new, 4, display.WHITE)
        display.display()

    def settle_level(first_level, settle_sec=0.3):
        # 0.3초 동안 값이 바뀌면 타이머 리셋 -> 최종 안정값 선택
        stable = first_level
        deadline = time() + settle_sec

        while time() < deadline:
            v = mainController.read_level()
            if v != stable:
                stable = v
                deadline = time() + settle_sec
            delay(20)

        return stable

def main():
    
    
    # 실행 시작할 때 초기화
    global app_running
    app_running = True   
    
    # 버튼 변수 초기화 #버튼 입출력 확인용
    prev_btn0 = True
    prev_btn1 = True

    mainController.draw_main_static()
    level = mainController.read_level()
    mainController.draw_cursor(None, level)

    while app_running: # 일반 True 대신 사용
        btn_0 = btn[0].read()
        btn_1 = btn[1].read()

        now_level = mainController.read_level()

        # poten 값 변동 시: 0.3초 안정화 후 원만 갱신
        if now_level != level:
            new_level = mainController.settle_level(now_level, 0.3)
            if new_level != level:
                mainController.draw_cursor(level, new_level)
                level = new_level

            # 안정화 중 눌린 버튼 입력은 무시
            prev_btn0 = btn[0].read()
            prev_btn1 = btn[1].read()
            delay(100) #0.1초 빠른 주기
            continue

        # 버튼 1회 입력(에지)
        if prev_btn0 and (not btn_0):
            run_action(SW0_ACTIONS, level)
            mainController.draw_main_static()
            mainController.draw_cursor(None, level)
            prev_btn0 = btn[0].read()
            prev_btn1 = btn[1].read()

        elif prev_btn1 and (not btn_1):
            run_action(SW1_ACTIONS, level)
            mainController.draw_main_static()
            mainController.draw_cursor(None, level)
            prev_btn0 = btn[0].read()
            prev_btn1 = btn[1].read()

        else:
            prev_btn0 = btn_0
            prev_btn1 = btn_1

        delay(300)

if __name__ == "__main__":
    main()



SW0 -> Night
mode -> Led All On
mode -> Led Siren
mode -> Led Wave
mode -> Led to Poten
mode -> led to Psd


KeyboardInterrupt: 